# 01 · Running the experiment

**Your job in one line: produce the numbers the paper reports.**

You do not need to understand the agent's internals to do this well. You need to
run it correctly, repeatedly, and record what you ran.

Background first, if you want it:
`../rashan-table-agent/01_the_data.ipynb` and `04_the_agent.ipynb`.

> **Note on saved outputs.** The cells that call a model are left unexecuted in
> this commit — the OpenAI account was at its daily request cap when it was
> written. They will run normally with your own key. Step 1 (the provider check)
> ran and its real output is saved.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `run_eval.py` | runs configurations and scores them against gold | `--configs baseline,cot,agent` |
| `--repeats` | how many times to run everything; 3 is the minimum for a spread | `--repeats 3` |
| `--sample-seed` | take a representative random sample, not the first N | `--sample-seed 7` |
| `--grounded-only` | keep only items whose gold is present in the table | flag |
| `BUDGET_USD` | hard spend cap; stops and saves partial results | `BUDGET_USD=2` |

## Where to run what

```
 JUPYTER    a small smoke test (2-5 questions) to confirm your key works,
            and all the analysis (notebooks 02 and 03).
 TERMINAL   the full run. It streams progress, survives, and is easy to
            restart. A notebook buffers subprocess output, so you would stare
            at a blank cell, and a kernel restart loses the whole run.
```

Rule of thumb: **use the notebook to check something works; use the terminal to
let it run.**

## Step 1 — confirm your key is loaded

Never start a long run without checking which model you are about to pay for.

In [1]:
import os, sys
sys.path.insert(0, "../rashan-table-agent")
import agent as base
base._load_env_file()
p = base.resolve_provider()
print("provider :", p)
print("model    :", base.OPENAI_MODEL if p == "openai" else base.GROQ_MODEL)
print("ceiling  : $", os.environ.get("BUDGET_USD", "2.0 (default)"))

provider : openai
model    : gpt-4o-mini
ceiling  : $ 2.0 (default)


## Step 2 — smoke test: one real question

If this prints a sensible answer, your key and the whole pipeline work. Do this
before launching anything long.

In [ ]:
import scitableqa_loader as L, scorer
t = [x for x in L.load_domain("Biology") if x.grounded][0]
out = base.answer(t.question, t.table_csv, False, 0.0)
print("Q       :", t.question[:84])
print("answer  :", repr(out["final"]), "| gold:", t.gold)
print("correct :", scorer.is_correct(out["final"], t.gold, t.question))
print("tokens  :", out["prompt_tokens"], "in /", out["completion_tokens"], "out")

## Step 3 — a 4-question end-to-end check

Confirms the harness, the scorer and the cost meter all work together. Takes
well under a minute and costs a fraction of a cent.

In [ ]:
import subprocess
cmd = [sys.executable, "run_eval.py", "--configs", "baseline,agent",
       "--limit", "4", "--sample-seed", "7", "--grounded-only"]
r = subprocess.run(cmd, capture_output=True, text=True,
                   cwd="../rashan-table-agent",
                   env={**os.environ, "BUDGET_USD": "2"})
print(r.stdout[-1800:])
if r.returncode: print("STDERR:", r.stderr[-500:])

## Step 4 — the real run, in a TERMINAL

From `02-experiments/rashan-table-agent/`:

```bash
# TASK 1 - main run (Biology)
BUDGET_USD=2 python run_eval.py --configs baseline,cot,agent \
    --limit 50 --repeats 3 --sample-seed 7 --grounded-only \
    2>&1 | tee results/run_biology.log

# TASK 4 - transfer arm (CompSci)
BUDGET_USD=2 python run_eval.py --domain CompSci --configs baseline,agent \
    --limit 50 --repeats 3 --sample-seed 7 --grounded-only \
    2>&1 | tee results/run_compsci.log
```

Why these flags:

```
 --repeats 3      a spread is the variation BETWEEN runs. With one run there is
                  nothing to vary, and step 2 asks for "mean +/- spread".
 --sample-seed 7  file order groups questions by source table, so the first 50
                  can all come from one table and misrepresent the benchmark.
 --grounded-only  keeps items whose gold answer is actually present in the table.
 tee              keeps a log you can paste if something goes wrong.
```

Both runs together cost well under $1 on `gpt-4o-mini`.

## Step 5 — before you re-run anything

**`run_eval.py` overwrites `results/*.csv`.** Copy them aside or commit them
first, or the previous run is gone.

```bash
cp -r results results_biology_run1
```

## Step 6 — record every run

Put this in `results/RUN_LOG.md`. A number nobody can trace is not usable in a
paper.

```
 date | the command exactly as typed | model | N questions | repeats
      | total spend printed at the end | anything that went wrong
```

If a run stops early, that is the budget ceiling doing its job: it saves what
finished and prints where it stopped. Record that number and tell Rashan rather
than raising the ceiling and rerunning blindly.

## What "done" looks like

```
 · results/*.csv committed for every configuration and repeat
 · results/RUN_LOG.md filled in
 · the spend reported for each run
```

Next: `02_analyse_results.ipynb`.